# PyTorch Fundamentals

**Module 01: Python for AI**  
**Objective**: Master PyTorch tensors, autograd, and neural network basics

PyTorch is the most popular deep learning framework for research and increasingly for production. Understanding PyTorch deeply means:
- Efficient tensor operations on GPU
- Automatic differentiation (autograd)
- Building custom neural network architectures
- Debugging gradient flow
- Mixed precision training

## What You'll Learn
1. Tensors vs NumPy arrays
2. GPU acceleration
3. Automatic differentiation (autograd)
4. Building neural networks with nn.Module
5. Custom layers and loss functions
6. Training loops
7. Mixed precision and performance optimization

## 1. Tensors: The Foundation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

### 1.1 Creating Tensors

In [ ]:
# From Python lists
x = torch.tensor([1, 2, 3, 4, 5])
print(f"1D tensor: {x}")
print(f"Shape: {x.shape}, Dtype: {x.dtype}, Device: {x.device}\n")

# 2D tensors
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f"2D tensor:\n{x}")
print(f"Shape: {x.shape}, Dtype: {x.dtype}\n")

# Common creation functions
zeros = torch.zeros(3, 4)
ones = torch.ones(2, 3)
rand = torch.rand(2, 3)  # Uniform [0, 1)
randn = torch.randn(2, 3)  # Normal(0, 1)
arange = torch.arange(0, 10, 2)
linspace = torch.linspace(0, 1, 5)

print(f"Zeros:\n{zeros}\n")
print(f"Ones:\n{ones}\n")
print(f"Random uniform:\n{rand}\n")
print(f"Random normal:\n{randn}\n")
print(f"Arange: {arange}")
print(f"Linspace: {linspace}\n")

# From NumPy (shares memory!)
np_array = np.array([1, 2, 3, 4])
torch_tensor = torch.from_numpy(np_array)
print(f"From NumPy: {torch_tensor}")

# Modifying torch_tensor modifies np_array!
torch_tensor[0] = 999
print(f"After modification - NumPy: {np_array}, Torch: {torch_tensor}\n")

# To NumPy (shares memory)
torch_tensor = torch.tensor([1, 2, 3, 4])
np_array = torch_tensor.numpy()
np_array[0] = 777
print(f"After NumPy modification - Torch: {torch_tensor}, NumPy: {np_array}")

### 1.2 GPU Acceleration

In [ ]:
# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Move tensor to GPU
x_cpu = torch.randn(3, 4)
print(f"CPU tensor: {x_cpu.device}")

if torch.cuda.is_available():
    x_gpu = x_cpu.to(device)
    # or x_gpu = x_cpu.cuda()
    print(f"GPU tensor: {x_gpu.device}\n")
    
    # Operations on GPU
    y_gpu = x_gpu * 2
    print(f"GPU operation result device: {y_gpu.device}\n")
    
    # Move back to CPU
    y_cpu = y_gpu.to('cpu')
    # or y_cpu = y_gpu.cpu()
    print(f"Back to CPU: {y_cpu.device}\n")

# Create directly on GPU
if torch.cuda.is_available():
    x = torch.randn(3, 4, device='cuda')
    print(f"Created directly on GPU: {x.device}")

# Performance comparison
size = 10000
x_cpu = torch.randn(size, size)

if torch.cuda.is_available():
    x_gpu = torch.randn(size, size, device='cuda')
    
    # CPU matmul
    start = time.time()
    y_cpu = x_cpu @ x_cpu
    cpu_time = time.time() - start
    
    # GPU matmul (with warmup)
    _ = x_gpu @ x_gpu  # Warmup
    torch.cuda.synchronize()
    start = time.time()
    y_gpu = x_gpu @ x_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    
    print(f"\nMatrix multiplication ({size}×{size}):")
    print(f"CPU time: {cpu_time:.4f}s")
    print(f"GPU time: {gpu_time:.4f}s")
    print(f"Speedup: {cpu_time/gpu_time:.1f}x")

## 2. Automatic Differentiation (Autograd)

The killer feature of PyTorch: automatic gradient computation.

In [ ]:
# Requires_grad tells PyTorch to track operations
x = torch.tensor([2.0], requires_grad=True)
print(f"x = {x}, requires_grad = {x.requires_grad}\n")

# Forward pass
y = x ** 2 + 3 * x + 1
print(f"y = x² + 3x + 1 = {y.item():.4f}\n")

# Backward pass (compute gradients)
y.backward()
print(f"dy/dx at x=2: {x.grad}")
print(f"Analytical: 2x + 3 = 2(2) + 3 = {2*2 + 3}\n")

# Multi-variable example
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x.sum() ** 2
print(f"y = (x1 + x2 + x3)² = {y.item()}")

y.backward()
print(f"Gradients: {x.grad}")
print(f"Analytical: dy/dx_i = 2(x1+x2+x3) = {2 * x.sum().item()}\n")

# Gradient accumulation (careful!)
x = torch.tensor([2.0], requires_grad=True)
for i in range(3):
    y = x ** 2
    y.backward()
    print(f"Iteration {i}: grad = {x.grad}")

print("\nGradients accumulate! Use x.grad.zero_() to reset\n")

# Proper way
x = torch.tensor([2.0], requires_grad=True)
for i in range(3):
    y = x ** 2
    y.backward()
    print(f"Iteration {i}: grad = {x.grad}")
    x.grad.zero_()  # Reset gradients

### 2.1 Computational Graph and Backpropagation

In [ ]:
# PyTorch builds a computational graph dynamically
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Forward: y = w*x + b
y = w * x + b
print(f"y = {y.item()}\n")

# Check grad_fn (shows computation history)
print(f"y.grad_fn: {y.grad_fn}")
print(f"w.grad_fn: {w.grad_fn} (leaf node)\n")

# Backward
y.backward()
print(f"dy/dx = {x.grad} (expected: w = 3)")
print(f"dy/dw = {w.grad} (expected: x = 2)")
print(f"dy/db = {b.grad} (expected: 1)\n")

# More complex example: neural network forward pass
x = torch.randn(32, 10, requires_grad=True)  # Batch of 32, 10 features
W1 = torch.randn(10, 20, requires_grad=True)
W2 = torch.randn(20, 1, requires_grad=True)

# Forward
h = torch.relu(x @ W1)
y_pred = h @ W2
loss = (y_pred ** 2).mean()

print(f"Loss: {loss.item():.4f}")

# Backward
loss.backward()
print(f"\nGradients computed:")
print(f"x.grad.shape: {x.grad.shape}")
print(f"W1.grad.shape: {W1.grad.shape}")
print(f"W2.grad.shape: {W2.grad.shape}")

### 2.2 No Grad Context (Inference)

In [ ]:
# During inference, we don't need gradients
x = torch.randn(100, 10, requires_grad=True)
w = torch.randn(10, 5, requires_grad=True)

# With gradients (training)
y_train = x @ w
print(f"Training mode - y.requires_grad: {y_train.requires_grad}\n")

# Without gradients (inference)
with torch.no_grad():
    y_eval = x @ w
    print(f"Inference mode - y.requires_grad: {y_eval.requires_grad}\n")

# Or use torch.inference_mode() (faster, more restrictive)
with torch.inference_mode():
    y_eval2 = x @ w
    print(f"Inference mode (v2) - y.requires_grad: {y_eval2.requires_grad}\n")

# Detach (create a new tensor without gradient tracking)
y_detached = (x @ w).detach()
print(f"Detached - y.requires_grad: {y_detached.requires_grad}")

## 3. Building Neural Networks with nn.Module

In [ ]:
# Simple linear regression
class LinearRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
    
    def forward(self, x):
        return self.linear(x)

model = LinearRegression(10, 1)
print(f"Model:\n{model}\n")

# Check parameters
print("Parameters:")
for name, param in model.named_parameters():
    print(f"{name}: shape={param.shape}, requires_grad={param.requires_grad}")

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"\nModel on: {next(model.parameters()).device}")

### 3.1 Multi-Layer Perceptron (MLP)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.1):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Hidden layers
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Create model
model = MLP(input_dim=784, hidden_dims=[512, 256, 128], output_dim=10)
print(f"MLP:\n{model}\n")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}\n")

# Forward pass
x = torch.randn(32, 784)  # Batch of 32 MNIST images
y = model(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {y.shape}")

### 3.2 Custom Layers

In [ ]:
class CustomLinear(nn.Module):
    """Linear layer implemented from scratch."""
    
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        
        # Initialize parameters
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)
    
    def forward(self, x):
        # x: (batch_size, in_features)
        output = x @ self.weight.T  # (batch_size, out_features)
        
        if self.bias is not None:
            output = output + self.bias
        
        return output
    
    def extra_repr(self):
        return f'in_features={self.weight.shape[1]}, out_features={self.weight.shape[0]}, bias={self.bias is not None}'

# Test
custom = CustomLinear(10, 5)
builtin = nn.Linear(10, 5)

x = torch.randn(32, 10)
y_custom = custom(x)
y_builtin = builtin(x)

print(f"Custom: {y_custom.shape}")
print(f"Built-in: {y_builtin.shape}")
print(f"\nCustom layer:\n{custom}")

### 3.3 Residual Connections

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block: y = F(x) + x"""
    
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim)
        )
    
    def forward(self, x):
        return self.net(x) + x  # Residual connection

# Stack residual blocks
class ResNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_blocks, output_dim):
        super().__init__()
        
        self.embedding = nn.Linear(input_dim, hidden_dim)
        
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, hidden_dim * 2)
            for _ in range(num_blocks)
        ])
        
        self.head = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.embedding(x)
        
        for block in self.blocks:
            x = block(x)
        
        return self.head(x)

model = ResNet(input_dim=100, hidden_dim=256, num_blocks=4, output_dim=10)
x = torch.randn(16, 100)
y = model(x)
print(f"ResNet output: {y.shape}")
print(f"\nModel:\n{model}")

## 4. Loss Functions and Optimization

In [ ]:
# Classification loss
logits = torch.randn(32, 10)  # 32 samples, 10 classes
labels = torch.randint(0, 10, (32,))  # Ground truth labels

# Cross-entropy loss
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)
print(f"Cross-entropy loss: {loss.item():.4f}\n")

# Manual cross-entropy
log_probs = F.log_softmax(logits, dim=1)
manual_loss = F.nll_loss(log_probs, labels)
print(f"Manual cross-entropy: {manual_loss.item():.4f}\n")

# Regression loss
predictions = torch.randn(32, 1)
targets = torch.randn(32, 1)

mse_loss = F.mse_loss(predictions, targets)
mae_loss = F.l1_loss(predictions, targets)
huber_loss = F.smooth_l1_loss(predictions, targets)

print(f"MSE loss: {mse_loss.item():.4f}")
print(f"MAE loss: {mae_loss.item():.4f}")
print(f"Huber loss: {huber_loss.item():.4f}\n")

# Optimizer
model = nn.Linear(10, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print("Optimizers:")
print(f"SGD: {optimizer}\n")

# Other optimizers
adam = torch.optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))
adamw = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
rmsprop = torch.optim.RMSprop(model.parameters(), lr=0.01)

print(f"Adam: {adam}")
print(f"AdamW: {adamw}")
print(f"RMSprop: {rmsprop}")

## 5. Complete Training Loop

In [ ]:
# Synthetic dataset
torch.manual_seed(42)
X_train = torch.randn(1000, 20)
y_train = (X_train.sum(dim=1) > 0).long()  # Binary classification

X_val = torch.randn(200, 20)
y_val = (X_val.sum(dim=1) > 0).long()

# Model
model = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 2)
)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 20
batch_size = 32

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    
    for i in range(0, len(X_train), batch_size):
        # Get batch
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]
        
        # Forward pass
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= (len(X_train) / batch_size)
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = criterion(val_logits, y_val).item()
        
        val_preds = val_logits.argmax(dim=1)
        val_acc = (val_preds == y_val).float().mean().item()
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")

print("\nTraining complete!")

## 6. Mixed Precision Training

Use FP16 for faster training with less memory.

In [ ]:
# Check if mixed precision is available
print(f"Mixed precision supported: {torch.cuda.is_available()}\n")

if torch.cuda.is_available():
    from torch.cuda.amp import autocast, GradScaler
    
    model = MLP(input_dim=784, hidden_dims=[512, 256], output_dim=10).cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()
    
    # Dummy data
    X = torch.randn(64, 784, device='cuda')
    y = torch.randint(0, 10, (64,), device='cuda')
    
    # Training step with mixed precision
    optimizer.zero_grad()
    
    # Forward pass in FP16
    with autocast():
        logits = model(X)
        loss = criterion(logits, y)
    
    # Backward pass (scales loss to prevent underflow)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    print(f"Loss: {loss.item():.4f}")
    print("Mixed precision training successful!")
else:
    print("Mixed precision requires CUDA. Running on CPU.")

## 7. Gradient Clipping (Prevent Exploding Gradients)

In [ ]:
model = MLP(input_dim=100, hidden_dims=[256, 128], output_dim=10)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Dummy forward pass
X = torch.randn(32, 100)
y = torch.randint(0, 10, (32,))
logits = model(X)
loss = nn.CrossEntropyLoss()(logits, y)

# Backward
optimizer.zero_grad()
loss.backward()

# Check gradient norms
total_norm = 0.0
for p in model.parameters():
    if p.grad is not None:
        param_norm = p.grad.data.norm(2)
        total_norm += param_norm.item() ** 2
total_norm = total_norm ** 0.5
print(f"Gradient norm before clipping: {total_norm:.4f}")

# Clip gradients
max_norm = 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

# Check after clipping
total_norm_after = 0.0
for p in model.parameters():
    if p.grad is not None:
        param_norm = p.grad.data.norm(2)
        total_norm_after += param_norm.item() ** 2
total_norm_after = total_norm_after ** 0.5
print(f"Gradient norm after clipping: {total_norm_after:.4f}")

optimizer.step()

## 8. Saving and Loading Models

In [ ]:
# Save model
model = MLP(input_dim=784, hidden_dims=[512, 256], output_dim=10)
torch.save(model.state_dict(), 'model.pth')
print("Model saved to model.pth\n")

# Load model
model_loaded = MLP(input_dim=784, hidden_dims=[512, 256], output_dim=10)
model_loaded.load_state_dict(torch.load('model.pth'))
model_loaded.eval()
print("Model loaded from model.pth\n")

# Save entire checkpoint
checkpoint = {
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 0.123,
}
torch.save(checkpoint, 'checkpoint.pth')
print("Checkpoint saved\n")

# Load checkpoint
checkpoint = torch.load('checkpoint.pth')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']
print(f"Checkpoint loaded: epoch={epoch}, loss={loss}")

## 9. Debugging: Common Issues

In [ ]:
# Issue 1: Gradient not flowing
x = torch.tensor([2.0], requires_grad=True)
y = x.detach() * 2  # Detach breaks gradient
# y.backward()  # Error: element 0 of tensors does not require grad

# Issue 2: In-place operations
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
# x += 1  # In-place modification of leaf variable (error!)
# y.backward()

# Issue 3: Device mismatch
if torch.cuda.is_available():
    model = nn.Linear(10, 5).cuda()
    x = torch.randn(32, 10)  # CPU tensor
    # y = model(x)  # Error: input and weight on different devices
    
    # Fix: move input to same device
    y = model(x.cuda())
    print("Device mismatch resolved")

# Issue 4: Forgetting to zero gradients
model = nn.Linear(10, 5)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
x = torch.randn(32, 10)
target = torch.randn(32, 5)

for i in range(3):
    y = model(x)
    loss = ((y - target) ** 2).mean()
    loss.backward()
    print(f"Iteration {i}: grad norm = {model.weight.grad.norm().item():.2f}")
    # optimizer.zero_grad()  # Forgot to zero! Gradients accumulate

print("\nRemember to call optimizer.zero_grad()!")

## 10. Performance Tips

1. **Use GPU**: 10-100x speedup for large models
2. **Batch operations**: Process multiple samples together
3. **Mixed precision**: FP16 for 2x memory savings and speedup
4. **Pin memory**: `pin_memory=True` in DataLoader for faster CPU→GPU transfer
5. **Gradient accumulation**: Simulate larger batch sizes
6. **Gradient checkpointing**: Trade compute for memory
7. **torch.compile()** (PyTorch 2.0+): JIT compilation for speedup
8. **Profile**: Use `torch.profiler` to find bottlenecks

## Exercises

### Exercise 1: Implement Batch Normalization Layer
Create a custom BatchNorm layer with learnable γ and β.

In [ ]:
class CustomBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        # TODO: Initialize parameters
        pass
    
    def forward(self, x):
        # TODO: Implement forward pass
        # Training: use batch statistics
        # Eval: use running statistics
        pass

# Test
# bn = CustomBatchNorm1d(64)
# x = torch.randn(32, 64)
# y = bn(x)
# print(f"Output shape: {y.shape}")

### Exercise 2: Implement Learning Rate Scheduler
Create a custom cosine annealing scheduler.

In [ ]:
class CosineAnnealingLR:
    def __init__(self, optimizer, T_max, eta_min=0):
        """
        Args:
            optimizer: PyTorch optimizer
            T_max: Maximum number of iterations
            eta_min: Minimum learning rate
        """
        # TODO: Implement
        pass
    
    def step(self):
        # TODO: Update learning rate
        pass

# Test
# model = nn.Linear(10, 5)
# optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
# scheduler = CosineAnnealingLR(optimizer, T_max=100)
# 
# for epoch in range(100):
#     scheduler.step()
#     print(f"Epoch {epoch}: LR = {optimizer.param_groups[0]['lr']:.6f}")

### Exercise 3: Implement Attention Layer
Build a scaled dot-product attention layer with multi-head support.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        # TODO: Implement
        pass
    
    def forward(self, query, key, value, mask=None):
        # TODO: Implement forward pass
        pass

# Test
# attn = MultiHeadAttention(d_model=512, num_heads=8)
# q = torch.randn(32, 10, 512)  # (batch, seq_len, d_model)
# k = torch.randn(32, 10, 512)
# v = torch.randn(32, 10, 512)
# output = attn(q, k, v)
# print(f"Output shape: {output.shape}")

## Summary

You've mastered:
- ✅ Tensors and GPU acceleration
- ✅ Automatic differentiation (autograd)
- ✅ Building neural networks with nn.Module
- ✅ Custom layers and residual connections
- ✅ Training loops and optimization
- ✅ Mixed precision training
- ✅ Model saving/loading
- ✅ Debugging gradient flow

**Next**: Machine Learning algorithms from scratch!